### Cross-Border Logistics Distribution

A logistics company needs to transport cargo from a central hub to a regional warehouse. They are moving two types of goods: Standard Freight and Express Freight. 

1. Objective Function: Minimize the total transportation cost ($Z$), where Standard Freight costs \\$40 per ton and Express Freight costs \\$70 per ton:

$$\text{minimize} \; Z = 40x_S + 70x_E$$

2. Standard Constraints
* Minimum Demand: The warehouse requires at least 50 tons of total cargo.
$$x_S + x_E \ge 50$$
* Express Requirement: Out of the total cargo, at least 15 tons must be Express Freight.
$$x_E \ge 15$$
* Weight Capacity: The total weight cannot exceed 120 tons due to bridge weight limits.
$$x_S + x_E \le 120$$
3. The Either-Or Constraint (Shipping Lane Regulation)
The company must choose between Route A (Heavy Rail) or Route B (Air Cargo Bypass). Safety and licensing laws state they can use either route, but they must fully comply with the chosen route's restrictions:
* Option 1 (Route A Restrictions): Total volume is strictly limited. Standard freight takes up more space, so the combined weighted volume limit is tight.
$$2x_S + x_E \le 100$$
* Option 2 (Route B Restrictions): Standard freight is heavily restricted to prevent blocking emergency air lanes.$$x_S \le 20$$

#### Define the model name
Import `Model` from `docplex.mp.model` and create a `Model` object named <code>cross-border_logistics</code>.

In [11]:
from docplex.mp.model import Model
mdl = Model(name='cross-border_logistics')

#### Define the Parameters

In [12]:
COST_S = 40                                                                                                                                                                                                                                       
COST_E = 70                                                                                                                                                                                                                                       
MIN_TOTAL = 50                                                                                                                                                                                                                                    
MIN_EXPRESS = 15                                                                                                                                                                                                                                  
MAX_WEIGHT = 120 

#### Define the Decision Variables
Continuous Variables

In [13]:
x_S = mdl.continuous_var(name="Standard_Freight", lb=0)                                                                                                                                                                                           
x_E = mdl.continuous_var(name="Express_Freight", lb=0) 

Binary variable for route selection

In [14]:
y = mdl.binary_var(name="Route_Selection")    

#### Define the Objective Function

In [15]:
total_cost = (COST_S * x_S) + (COST_E * x_E)                                                                                                                                                                                                      
mdl.minimize(total_cost)  

#### Define the Constraints
Global Constraints

In [16]:
mdl.add_constraint(x_S + x_E >= MIN_TOTAL, "Min_Demand")                                                                                                                                                                                          
mdl.add_constraint(x_E >= MIN_EXPRESS, "Min_Express")                                                                                                                                                                                             
mdl.add_constraint(x_S + x_E <= MAX_WEIGHT, "Weight_Capacity")   

docplex.mp.LinearConstraint[Weight_Capacity](Standard_Freight+Express_Freight,LE,120)

Either-Or (Disjunctive) Constraints: 
* If Route A is chosen (y = 1), enforce Route A restrictions
* If Route B is chosen (y = 0), enforce Route B restrictions 

In [17]:
mdl.add_indicator(                                                                                                                                                                                                                                
    y,                                                                                                                                                                                                                                            
    2 * x_S + x_E <= 100,                                                                                                                                                                                                                         
    active_value=1,                                                                                                                                                                                                                               
    name="Route_A_Restrictions"                                                                                                                                                                                                                   
)                                                                                                                                                                                                                                                 
                                                                                                                                                                                                                                                    
mdl.add_indicator(                                                                                                                                                                                                                                
    y,                                                                                                                                                                                                                                            
    x_S <= 20,                                                                                                                                                                                                                                    
    active_value=0,                                                                                                                                                                                                                               
    name="Route_B_Restrictions"                                                                                                                                                                                                                   
)     

docplex.mp.constr.IndicatorConstraint[Route_B_Restrictions](Route_Selection,Standard_Freight<=20,true=0)

#### Solve the Model

In [18]:
solution = mdl.solve(log_output=True)      

Version identifier: 22.2.0.0 | 2026-07-01 | 7cae668b4
CPXPARAM_Read_DataCheck                          1
Tried aggregator 1 time.
MIP Presolve eliminated 1 rows and 0 columns.
Reduced MIP has 5 rows, 4 columns, and 11 nonzeros.
Reduced MIP has 1 binaries, 0 generals, 0 SOSs, and 2 indicators.
Presolve time = 0.01 sec. (0.01 ticks)
Found incumbent of value 7800.000000 after 0.01 sec. (0.01 ticks)
Probing fixed 0 vars, tightened 2 bounds.
Probing time = 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
MIP Presolve modified 3 coefficients.
Reduced MIP has 5 rows, 4 columns, and 11 nonzeros.
Reduced MIP has 1 binaries, 0 generals, 0 SOSs, and 2 indicators.
Presolve time = 0.00 sec. (0.00 ticks)
Probing time = 0.00 sec. (0.00 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 16 threads.
Root relaxation solution time = 0.00 sec. (0.01 ticks)

        Nodes                                         Cuts/
   Node 

#### Print Results

In [19]:
if solution:                                                                                                                                                                                                                                      
    print("\n=== OPTIMAL SOLUTION FOUND ===")                                                                                                                                                                                                     
    print(f"Status: {mdl.get_solve_status()}")                                                                                                                                                                                                    
    print(f"Total Transportation Cost: ${solution.objective_value:,.2f}")                                                                                                                                                                         
    print(f"Standard Freight: {solution[x_S]:.2f} tons")                                                                                                                                                                                          
    print(f"Express Freight: {solution[x_E]:.2f} tons")                                                                                                                                                                                           
                                                                                                                                                                                                                                                    
    if solution[y] == 1:                                                                                                                                                                                                                          
        print("Selected Route: Route A (Heavy Rail)")                                                                                                                                                                                             
    else:                                                                                                                                                                                                                                         
        print("Selected Route: Route B (Air Cargo Bypass)")                                                                                                                                                                                       
else:                                                                                                                                                                                                                                             
    print("\nNo optimal solution found.")  


=== OPTIMAL SOLUTION FOUND ===
Status: JobSolveStatus.OPTIMAL_SOLUTION
Total Transportation Cost: $2,450.00
Standard Freight: 35.00 tons
Express Freight: 15.00 tons
Selected Route: Route A (Heavy Rail)


### E-Commerce Smart Warehousing
A retail company is launching two new smartphone models: Model Alpha ($x_A$) and Model Beta ($x_B$). They need to determine how many units of each model to stock in their fulfillment center to maximize revenue. Model Alpha sells for \\$400 per unit and Model Beta sells for \\$600 per unit. The wholesale purchase cost is \\$200 for Alpha and \\$400 for Beta and the procurement budget is only \\$120,000.

1. Objective Function: Maximize total projected sales revenue (Z), where Model Alpha sells for \\$400 per unit and Model Beta sells for \\$600 per unit:
$$\text{maximize} \; Z = 400x_A + 600x_B$$ 
2. Standard Constraints
* Procurement Budget: The wholesale purchase cost is \\$200 for Alpha and \\$400 for Beta. The total spending limit is \\$120,000.
$$200x_A + 400x_B \le 120000$$ 
* Shelf Storage Volume: Alpha requires 1 slot of shelf space, and Beta requires 2 slots. The warehouse has a total allocation of 500 slots for these products.
$$x_A + 2x_B \le 500$$ 
3. The IF-THEN Condition (Market Saturation Clause)
The marketing department warns that the premium phone market is easily saturated. They enforce the following logic:
IF the company stocks a high volume of Model Beta ($x_B > 100$),
THEN they must limit the stock of Model Alpha to at most 180 units ($x_A \leq 180$) to avoid unsold inventory.

#### Define the model name
Import `Model` from `docplex.mp.model` and create a `Model` object named <code>e-commerce_warehousing</code.

In [26]:
from docplex.mp.model import Model                                                                                                                                                                                                                
                                                                                                                                                                                                                                                                                                                                                                                                                                                                             
mdl_ecom = Model(name="e-commerce_warehousing")    

#### Define the Parameters

In [27]:
BUDGET = 120000                                                                                                                                                                                                                                   
MAX_SHELF_SLOTS = 500  

#### Define the Decision Variables
Continuous Variables

In [28]:
x_A = mdl_ecom.continuous_var(name="Model_Alpha", lb=0)                                                                                                                                                                                           
x_B = mdl_ecom.continuous_var(name="Model_Beta", lb=0)  

Binary variable for route selection

In [29]:
y_ecom = mdl_ecom.binary_var(name="Beta_Exceed_100")                                                                                                                                                                                              
                                                        

#### Define the Objective Function

In [30]:
total_revenue = (400 * x_A) + (600 * x_B)                                                                                                                                                                                                         
mdl_ecom.maximize(total_revenue) 

#### Define the Constraints
Global Constraints

In [31]:
mdl_ecom.add_constraint(200 * x_A + 400 * x_B <= BUDGET, "Procurement_Budget")                                                                                                                                                                    
mdl_ecom.add_constraint(x_A + 2 * x_B <= MAX_SHELF_SLOTS, "Shelf_Storage_Volume")    

docplex.mp.LinearConstraint[Shelf_Storage_Volume](Model_Alpha+2Model_Beta,LE,500)

IF-THEN Constraint: IF the company stocks a high volume of Model Beta ($x_B > 100$),
THEN they must limit the stock of Model Alpha to at most 180 units ($x_A \leq 180$)
* if (y=0), then $x_B \leq 100$
* if (y=1), then $x_A \leq 180$

In [32]:
mdl_ecom.add_indicator(                                                                                                                                                                                                                           
    y_ecom,                                                                                                                                                                                                                                       
    x_B <= 100,                                                                                                                                                                                                                                   
    active_value=0,                                                                                                                                                                                                                               
    name="Link_y_zero"                                                                                                                                                                                                                            
)                                                                                                                                                                                                                                                 
                                                                                                                                                                                                                                                    
mdl_ecom.add_indicator(                                                                                                                                                                                                                           
    y_ecom,                                                                                                                                                                                                                                       
    x_A <= 180,                                                                                                                                                                                                                                   
    active_value=1,                                                                                                                                                                                                                               
    name="Market_Saturation_Clause"                                                                                                                                                                                                               
)    

docplex.mp.constr.IndicatorConstraint[Market_Saturation_Clause](Beta_Exceed_100,Model_Alpha<=180,true=1)

#### Solve the Model

In [33]:
solution_ecom = mdl_ecom.solve(log_output=True) 

Version identifier: 22.2.0.0 | 2026-07-01 | 7cae668b4
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 0.000000 after 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 1 rows and 0 columns.
Reduced MIP has 3 rows, 3 columns, and 6 nonzeros.
Reduced MIP has 1 binaries, 0 generals, 0 SOSs, and 2 indicators.
Presolve time = 0.00 sec. (0.00 ticks)
Probing time = 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
Reduced MIP has 3 rows, 3 columns, and 6 nonzeros.
Reduced MIP has 1 binaries, 0 generals, 0 SOSs, and 2 indicators.
Presolve time = 0.00 sec. (0.00 ticks)
Probing time = 0.00 sec. (0.00 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 16 threads.
Root relaxation solution time = 0.00 sec. (0.00 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    ItCnt     Gap

*     0+    0

#### Print Results

In [34]:
if solution_ecom:                                                                                                                                                                                                                                 
    print("\n=== OPTIMAL SOLUTION FOUND ===")                                                                                                                                                                                                     
    print(f"Status: {mdl_ecom.get_solve_status()}")                                                                                                                                                                                               
    print(f"Total Projected Sales Revenue: ${solution_ecom.objective_value:,.2f}")                                                                                                                                                                
    print(f"Model Alpha: {solution_ecom[x_A]:.0f} units")                                                                                                                                                                                         
    print(f"Model Beta: {solution_ecom[x_B]:.0f} units")                                                                                                                                                                                          
else:                                                                                                                                                                                                                                             
    print("\nNo optimal solution found.") 


=== OPTIMAL SOLUTION FOUND ===
Status: JobSolveStatus.OPTIMAL_SOLUTION
Total Projected Sales Revenue: $200,000.00
Model Alpha: 500 units
Model Beta: 0 units
